# 03. IsolationForest — `cal01_base_v1.csv` 센서별 X 실험 (A/B)

**목적:** 14개 LOT · 21개 센서의 **1초 원본**에서 고장 확정이 아닌 *검토 가치가 있는 특이 시점*을 탐색합니다.

- **A:** 센서 현재값 1개 → 모든 21개 센서
- **B:** 현재값, 초당 변화량, 최근 60초 평균 편차, 최근 60초 표준편차 → 모든 21개 센서
- 센서별·실험별로 서로 다른 IsolationForest를 학습합니다. 전 LOT에서 학습하므로 **미래 예측 성능 평가가 아닙니다**.
- `LOT_NO`, `GRADE`, `EXCLUDED`, `EXCLUSION_REASON`, `LABEL_STATUS`, `LABEL`, `MODEL_ELIGIBLE`, 조업·정비·판정 컬럼은 **모델 X에서 제외**합니다.
- 결측값은 보간하지 않고 점수 계산 불가(`IF_FLAG=-1`)로 기록합니다. LOT 경계에서 과거 이력이 다른 LOT로 넘어가지 않습니다.
- 서로 다른 실험의 상위 5% 판정은 후보 *양*이 비슷하므로 **후보가 얼마나 겹치고 어떻게 지속되는지** 비교하는 데 사용합니다.

**필요 패키지:** `pandas`, `numpy`, `matplotlib`, `scikit-learn`, `IPython`, `nbformat`(노트북 편집 시). 결과 CSV 7개와 그래프 PNG 모두 `data/saylestkin/isolation_forest_abc/` 폴더에 함께 저장합니다. 실행 위치에 따라 저장 경로가 바뀌지 않도록 프로젝트 폴더를 자동 탐색합니다.

In [ ]:
# 1. 라이브러리
from pathlib import Path
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import IsolationForest

if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# 2. 실행 설정 (입력/출력 경로)
# 현재 작업 위치가 pdm-project, notebooks, notebooks/sayletskin 중 어디여도
# cal01_base_v1.csv가 있는 프로젝트 폴더를 자동으로 찾습니다.
PROJECT_DIR = Path.cwd().resolve()
for _ in range(5):
    if (PROJECT_DIR / "data" / "cal01_base_v1.csv").exists():
        break
    PROJECT_DIR = PROJECT_DIR.parent

# 위에서 프로젝트 경로를 못 찾는 경우에만 아래 줄의 주석을 해제하고 직접 입력하세요.
# PROJECT_DIR = Path(r"C:\Users\tomrk\Desktop\PDM\pdm-project")

# 원본 CSV 입력 경로
DATA_PATH = PROJECT_DIR / "data" / "cal01_base_v1.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"원본 CSV를 찾을 수 없습니다: {DATA_PATH}\n"
        "PROJECT_DIR이 pdm-project 폴더인지 확인하세요."
    )

# 결과 CSV 7개: data/sayletskin/isolation_forest_abc 폴더에 저장
OUT_DIR = PROJECT_DIR / "data" / "sayletskin" / "isolation_forest_abc"

# 그래프 PNG도 CSV와 동일한 폴더에 저장합니다. (별도 figures 폴더 없음)
FIG_DIR = OUT_DIR

# 보고싶은 센서와 LOT을 지정 후 Save를 True로 설정
SAVE_CSV = False
SAVE_FIGURES = False
PLOT_ALL = False
FOCUS_SENSORS = ["OP-Z3", "CP-Z4"]
FOCUS_LOTS = ["240058"] 

# A/B 두 실험만 실행합니다. (C 실험 제외)
EXPERIMENTS = ["A", "B"]
FOCUS_ONLY = False
N_ESTIMATORS = 60
MAX_TRAIN_ROWS = 12000
MAX_SAMPLES = 256
RANDOM_STATE = 42
MIN_ROLLING_POINTS = 10
MAX_DIFF_GAP_SEC = 15
MERGE_GAPS = [0, 1, 2, 3, 4, 5]
TOP5_PERCENTILE = 0.95

# CSV 또는 그래프 저장을 사용할 때만 공통 폴더를 생성합니다.
if SAVE_CSV or SAVE_FIGURES:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

print("프로젝트:", PROJECT_DIR)
print("입력 CSV:", DATA_PATH)
print("결과 CSV 폴더:", OUT_DIR)
print("그래프 폴더:", FIG_DIR)


In [ ]:
# 3. 파일의 필요한 컬럼만 읽습니다. 106개 전체 컬럼을 메모리에 올리지 않습니다.
if not DATA_PATH.exists():
    raise FileNotFoundError(f"데이터 파일을 찾을 수 없습니다: {DATA_PATH}")

sensor_cols = (["CLN-IN"] + [f"TC-Z{i}" for i in range(1, 10)]
               + ["TC-OUT1", "TC-OUT2"] + [f"OP-Z{i}" for i in range(1, 9)]
               + ["CP-Z4"])
if FOCUS_ONLY:
    sensor_cols = [sensor for sensor in sensor_cols if sensor in FOCUS_SENSORS]

header = pd.read_csv(DATA_PATH, encoding="utf-8-sig", nrows=0).columns.tolist()
required = ["LOT_NO", "MEAS_DT"] + sensor_cols
missing = [c for c in required if c not in header]
if missing:
    raise ValueError(f"필수 컬럼이 없습니다: {missing}")

meta_cols = [c for c in ["ROW_ID", "LOT_NO", "MEAS_DT", "GRADE", "LOT_ELAPSED_MIN",
                         "EXCLUDED", "EXCLUSION_REASON", "LABEL_STATUS"] if c in header]
input_cols = list(dict.fromkeys(meta_cols + sensor_cols))
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig", usecols=input_cols,
                 dtype={"LOT_NO": "string"}, low_memory=False)
df["MEAS_DT"] = pd.to_datetime(df["MEAS_DT"], errors="coerce")
if df[["LOT_NO", "MEAS_DT"]].isna().any().any():
    raise ValueError("LOT_NO 또는 MEAS_DT에 해석 불가능한 값이 있습니다.")

df = df.sort_values(["LOT_NO", "MEAS_DT"], kind="stable").reset_index(drop=True)
for sensor in sensor_cols:
    df[sensor] = pd.to_numeric(df[sensor], errors="coerce")

lot_indices = {}
for lot, block in df.groupby("LOT_NO", sort=False):
    lot_indices[str(lot)] = block.index.to_numpy()

print("행 수:", len(df), "LOT:", len(lot_indices), "센서:", len(sensor_cols))
print("시간 중복:", int(df.duplicated(["LOT_NO", "MEAS_DT"]).sum()))
print("센서별 결측 상위:")
display(df[sensor_cols].isna().sum().sort_values(ascending=False).head(8).to_frame("결측 수"))

## 4. 파생변수 X를 계산하는 규칙

- **현재값** `VALUE`는 해당 센서의 실제 기록값입니다.
- **초당 변화량** `DIFF_PER_SEC`은 이전 측정과의 차이를 실제 경과 초로 나눕니다. 측정 간격이 0초이거나 15초 초과이면 계산하지 않습니다.
- **60초 평균 편차** `DEV_60S`와 **60초 표준편차** `STD_60S`는 `MEAS_DT`의 *직전 60초*만 사용합니다. 현재 시점은 계산에서 제외합니다.
- **A와 B의 유효 행 수는 다를 수 있습니다.** `07_if_overlap.csv`는 *두 실험 모두 점수화된 행*만 비교합니다.

In [ ]:
# 4. 실험별 결과 저장 공간
all_rows = df[meta_cols + sensor_cols].copy()
sensor_stats = []
threshold_rows = []
lot_summaries = []
candidate_parts = []

# 전체 21개 센서 × A/B 두 실험 = 총 42개 모델
experiment_plan = [(sensor, experiment) for sensor in sensor_cols for experiment in EXPERIMENTS]
print("총 학습 모델 수:", len(experiment_plan))


In [ ]:
# 5. 센서별 A/B 모델을 학습하고 전체 행에 점수를 계산합니다.
# X에 사용되는 컬럼은 VALUE/DIFF/DEV/STD뿐입니다.
for sensor_no, sensor in enumerate(sensor_cols, 1):
    print(f"[{sensor_no:02d}/{len(sensor_cols)}] {sensor} 파생변수 계산")

    values_all = df[sensor].to_numpy(dtype="float64")
    diff_all = np.full(len(df), np.nan, dtype="float32")
    dev_all = np.full(len(df), np.nan, dtype="float32")
    std_all = np.full(len(df), np.nan, dtype="float32")

    # LOT별로만 과거 60초를 참고합니다. 다른 LOT의 측정값은 사용하지 않습니다.
    for lot, idx in lot_indices.items():
        times = pd.DatetimeIndex(df.loc[idx, "MEAS_DT"])
        values = values_all[idx]
        past = pd.Series(values, index=times).rolling(
            "60s", min_periods=MIN_ROLLING_POINTS, closed="left")
        previous_mean = past.mean().to_numpy()
        previous_std = pd.Series(values, index=times).rolling(
            "60s", min_periods=MIN_ROLLING_POINTS, closed="left").std().to_numpy()

        seconds = pd.Series(times).diff().dt.total_seconds().to_numpy()
        change = np.full(len(idx), np.nan)
        usable = (seconds > 0) & (seconds <= MAX_DIFF_GAP_SEC)
        np.divide(np.diff(values, prepend=np.nan), seconds, out=change, where=usable)
        diff_all[idx] = change
        dev_all[idx] = values - previous_mean
        std_all[idx] = previous_std

    features = pd.DataFrame({
        "VALUE": values_all.astype("float32"),
        "DIFF_PER_SEC": diff_all,
        "DEV_60S": dev_all,
        "STD_60S": std_all,
    })
    for experiment in EXPERIMENTS:
        if experiment == "A":
            x_cols = ["VALUE"]
        elif experiment == "B":
            x_cols = ["VALUE", "DIFF_PER_SEC", "DEV_60S", "STD_60S"]

        valid = np.isfinite(features[x_cols].to_numpy()).all(axis=1)
        scored_rows = int(valid.sum())
        if scored_rows < 300:
            print(f"    {experiment}: 유효 행 {scored_rows}개 → 학습 생략")
            continue

        # 학습 데이터는 각 LOT에서 고르게 무작위 추출합니다 (A 프로젝트).
        rng = np.random.default_rng(RANDOM_STATE)
        train_idx = []
        per_lot = max(1, MAX_TRAIN_ROWS // len(lot_indices))
        for lot, idx in lot_indices.items():
            usable_idx = idx[valid[idx]]
            if len(usable_idx) == 0:
                continue
            picked = rng.choice(usable_idx, size=min(per_lot, len(usable_idx)), replace=False)
            train_idx.extend(picked.tolist())

        model = IsolationForest(
            n_estimators=N_ESTIMATORS, max_samples=MAX_SAMPLES,
            contamination="auto", random_state=RANDOM_STATE, n_jobs=2)
        model.fit(features.loc[train_idx, x_cols].to_numpy(dtype="float32"))

        # -score_samples : 클수록 특이함 / 미계산 행은 NaN
        score = np.full(len(df), np.nan, dtype="float32")
        score[valid] = -model.score_samples(
            features.loc[valid, x_cols].to_numpy(dtype="float32")).astype("float32")
        scored = score[valid]
        t_auto = float(-model.offset_)
        t_5 = float(np.quantile(scored, 0.95))
        t_2 = float(np.quantile(scored, 0.98))
        t_1 = float(np.quantile(scored, 0.99))
        all_same = bool(np.isclose(scored.min(), scored.max()))
        flag_5 = valid & (score >= t_5) & (not all_same)
        flag_2 = valid & (score >= t_2) & (not all_same)
        flag_1 = valid & (score >= t_1) & (not all_same)
        flag_auto = valid & (score > t_auto)

        # 1 = TOP5 후보 / 0 = 비후보 / -1 = 계산 불가
        all_rows[f"{sensor}__IF_SCORE_{experiment}"] = score
        all_rows[f"{sensor}__IF_FLAG_{experiment}"] = np.where(
            valid, flag_5.astype("int8"), -1).astype("int8")

        sensor_stats.append({
            "EXPERIMENT": experiment, "SENSOR": sensor, "X_COLUMNS": ", ".join(x_cols),
            "TOTAL_ROWS": len(df), "TRAIN_ROWS": len(train_idx),
            "SCORED_ROWS": scored_rows, "UNSCORED_ROWS": len(df) - scored_rows,
            "SCORE_MIN": float(scored.min()), "SCORE_MEAN": float(scored.mean()),
            "SCORE_MAX": float(scored.max()), "TOP5_COUNT": int(flag_5.sum()),
            "TOP2_COUNT": int(flag_2.sum()), "TOP1_COUNT": int(flag_1.sum()),
            "AUTO_COUNT": int(flag_auto.sum())})
        threshold_rows.append({
            "EXPERIMENT": experiment, "SENSOR": sensor,
            "AUTO_THRESHOLD": t_auto, "TOP5_THRESHOLD": t_5,
            "TOP2_THRESHOLD": t_2, "TOP1_THRESHOLD": t_1})

        for lot, idx in lot_indices.items():
            n_valid = int(valid[idx].sum())
            lot_summaries.append({
                "EXPERIMENT": experiment, "LOT_NO": lot,
                "GRADE": df.loc[idx[0], "GRADE"] if "GRADE" in df.columns else "",
                "SENSOR": sensor, "TOTAL_ROWS": len(idx), "SCORED_ROWS": n_valid,
                "TOP5_COUNT": int(flag_5[idx].sum()),
                "TOP5_RATE": round(float(flag_5[idx].sum()) / n_valid, 6) if n_valid else np.nan,
                "TOP2_COUNT": int(flag_2[idx].sum()), "TOP1_COUNT": int(flag_1[idx].sum())})

        candidates = df.loc[flag_5, meta_cols].copy()
        candidates["EXPERIMENT"] = experiment
        candidates["SENSOR"] = sensor
        candidates["SENSOR_VALUE"] = values_all[flag_5]
        candidates["IF_SCORE"] = score[flag_5]
        candidates["AUTO_FLAG"] = flag_auto[flag_5].astype("int8")
        candidates["TOP2_FLAG"] = flag_2[flag_5].astype("int8")
        candidates["TOP1_FLAG"] = flag_1[flag_5].astype("int8")
        candidate_parts.append(candidates)

        print(f"    {experiment}: X={len(x_cols)}개 / 학습={len(train_idx):,}행 / 점수화={scored_rows:,}행 / TOP5={int(flag_5.sum()):,}행")

print("전체 학습 완료")

In [ ]:
# 6. 기본 결과 CSV 5개를 저장합니다 (05_runs는 다음 셀).
df_stats = pd.DataFrame(sensor_stats)
df_thresholds = pd.DataFrame(threshold_rows)
df_lot_summary = pd.DataFrame(lot_summaries)
df_candidates = pd.concat(candidate_parts, ignore_index=True)
df_candidates = df_candidates.sort_values(
    ["EXPERIMENT", "LOT_NO", "SENSOR", "MEAS_DT"]).reset_index(drop=True)

if SAVE_CSV:
    df_stats.to_csv(OUT_DIR / "01_if_sensor_stats.csv", index=False, encoding="utf-8-sig")
    df_thresholds.to_csv(OUT_DIR / "02_if_thresholds.csv", index=False, encoding="utf-8-sig")
    df_lot_summary.to_csv(OUT_DIR / "03_if_lot_sensor_summary.csv", index=False, encoding="utf-8-sig")
    df_candidates.to_csv(OUT_DIR / "04_if_candidates.csv", index=False, encoding="utf-8-sig")
    all_rows.to_csv(OUT_DIR / "06_if_all_rows.csv", index=False,
                    encoding="utf-8-sig", float_format="%.7g")

print("모델별 요약:")
display(df_stats[["EXPERIMENT", "SENSOR", "SCORED_ROWS", "TOP5_COUNT"]])
print("TOP5 후보 행(실험×센서 기준):", len(df_candidates))

In [ ]:
# 7. 상위 5% 후보를 연속 구간으로 묶습니다 (간격 0~5초 비교).
# gap=0: 바로 다음 초에 후보가 나온 경우만 병합
# gap=5: 사이에 정상/결측 초가 최대 5초 있어도 한 구간으로 병합
run_path = OUT_DIR / "05_if_runs.csv"
run_cols = ["EXPERIMENT", "LOT_NO", "GRADE", "SENSOR", "MERGE_GAP_SEC",
            "START_DT", "END_DT", "DURATION_SEC", "DETECTED_ROWS",
            "OTHER_OR_MISSING_SEC", "PEAK_IF_SCORE"]
run_parts = []
for (experiment, sensor), group in df_candidates.groupby(["EXPERIMENT", "SENSOR"], sort=False):
    selected = group.sort_values(["LOT_NO", "MEAS_DT"]).copy()
    for gap in MERGE_GAPS:
        seconds = selected["MEAS_DT"].diff().dt.total_seconds()
        begins = selected["LOT_NO"].ne(selected["LOT_NO"].shift()) | (seconds > gap + 1)
        selected["RUN_ID"] = begins.cumsum()
        runs = selected.groupby("RUN_ID", sort=False).agg(
            LOT_NO=("LOT_NO", "first"),
            START_DT=("MEAS_DT", "first"), END_DT=("MEAS_DT", "last"),
            DETECTED_ROWS=("MEAS_DT", "size"),
            PEAK_IF_SCORE=("IF_SCORE", "max")).reset_index(drop=True)
        runs["EXPERIMENT"] = experiment
        runs["SENSOR"] = sensor
        runs["MERGE_GAP_SEC"] = gap
        runs["GRADE"] = selected.groupby("RUN_ID", sort=False)["GRADE"].first().to_numpy() if "GRADE" in selected else ""
        runs["DURATION_SEC"] = (runs["END_DT"] - runs["START_DT"]).dt.total_seconds().astype("int64") + 1
        runs["OTHER_OR_MISSING_SEC"] = runs["DURATION_SEC"] - runs["DETECTED_ROWS"]
        run_parts.append(runs[run_cols])

df_runs = pd.concat(run_parts, ignore_index=True) if run_parts else pd.DataFrame(columns=run_cols)
if SAVE_CSV:
    df_runs.to_csv(run_path, index=False, encoding="utf-8-sig")
print("연속 구간 요약:")
display(df_runs.groupby(["EXPERIMENT", "MERGE_GAP_SEC"]).size().unstack(fill_value=0))

In [ ]:
# 8. 동일 센서·시각에서 실험별 상위 5% 후보의 겹침을 비교합니다.
# 두 모델 모두 점수가 계산된 행만 비교합니다.
overlap_rows = []
for sensor in sensor_cols:
    pairs = [("A", "B")]
    for left, right in pairs:
        col_a = f"{sensor}__IF_FLAG_{left}"
        col_b = f"{sensor}__IF_FLAG_{right}"
        if col_a not in all_rows or col_b not in all_rows:
            continue
        flags_a = all_rows[col_a].to_numpy()
        flags_b = all_rows[col_b].to_numpy()
        shared = (flags_a >= 0) & (flags_b >= 0)
        for lot, idx in lot_indices.items():
            shared_lot = shared[idx]
            a_lot = (flags_a[idx] == 1) & shared_lot
            b_lot = (flags_b[idx] == 1) & shared_lot
            both = a_lot & b_lot
            either = a_lot | b_lot
            overlap_rows.append({
                "SENSOR": sensor, "LOT_NO": lot, "LEFT": left, "RIGHT": right,
                "SHARED_SCORABLE_ROWS": int(shared_lot.sum()),
                "BOTH": int(both.sum()), "LEFT_ONLY": int((a_lot & ~b_lot).sum()),
                "RIGHT_ONLY": int((b_lot & ~a_lot).sum()),
                "UNION": int(either.sum()),
                "JACCARD": float(both.sum() / either.sum()) if either.any() else np.nan})

df_overlap = pd.DataFrame(overlap_rows)
if SAVE_CSV:
    df_overlap.to_csv(OUT_DIR / "07_if_overlap.csv", index=False, encoding="utf-8-sig")
print("LOT 240058, OP-Z3 실험 간 겹침(예시):")
display(df_overlap[(df_overlap["LOT_NO"] == "240058") & (df_overlap["SENSOR"] == "OP-Z3")])

In [ ]:
# 9. 그래프 저장: 기본은 LOT 240058의 OP-Z3, CP-Z4 A/B 비교
if SAVE_FIGURES:
    plot_sensors = sensor_cols if PLOT_ALL else [s for s in FOCUS_SENSORS if s in sensor_cols]
    plot_lots = list(lot_indices) if PLOT_ALL else [l for l in FOCUS_LOTS if l in lot_indices]
    t_map = df_thresholds.set_index(["SENSOR", "EXPERIMENT"])["TOP5_THRESHOLD"].to_dict()
    figure_count = 0
    for sensor in plot_sensors:
        for experiment in EXPERIMENTS:
            score_col = f"{sensor}__IF_SCORE_{experiment}"
            flag_col = f"{sensor}__IF_FLAG_{experiment}"
            if score_col not in all_rows:
                continue
            for lot in plot_lots:
                idx = lot_indices[lot]
                t = df.loc[idx, "MEAS_DT"]
                vals = df.loc[idx, sensor]
                scores = all_rows.loc[idx, score_col]
                flags = (all_rows.loc[idx, flag_col] == 1).to_numpy()

                fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
                axes[0].plot(t, vals, linewidth=0.9, label=sensor)
                axes[0].scatter(t.iloc[np.flatnonzero(flags)], vals.iloc[np.flatnonzero(flags)],
                                s=8, c="crimson", label="TOP5", zorder=3)
                axes[0].set_ylabel("센서값")
                axes[0].legend()
                axes[1].plot(t, scores, linewidth=0.8, label="IF_SCORE")
                axes[1].axhline(t_map[(sensor, experiment)], color="crimson", linestyle="--", label="상위 5% 임계")
                axes[1].set_ylabel("이상점수")
                axes[1].set_xlabel("MEAS_DT")
                axes[1].legend()
                fig.suptitle(f"LOT {lot} | {sensor} | IsolationForest {experiment} (1초 원본)")
                fig.autofmt_xdate()
                fig.tight_layout()
                fig.savefig(FIG_DIR / f"if_{experiment}_{lot}_{sensor}.png", dpi=120, bbox_inches="tight")
                plt.close(fig)
                figure_count += 1
    print("그림 저장:", figure_count, "개 /", FIG_DIR.resolve())
else:
    print("SAVE_FIGURES=False: 그래프 생략")

## 10. 출력 파일 안내

| 파일 | 의미 |
| --- | --- |
| `01_if_sensor_stats.csv` | A/B 센서별 학습·점수화 행 수와 후보 건수 |
| `02_if_thresholds.csv` | 센서별·실험별 `auto` / 상위 5·2·1% 임계값 |
| `03_if_lot_sensor_summary.csv` | LOT별 센서별 탐지 수와 비율 |
| `04_if_candidates.csv` | TOP5 후보 행, 실험명·센서명·이상점수 포함 |
| `05_if_runs.csv` | 0~5초 공백 병합별 연속 구간 및 탐지 실제 행 수 |
| `06_if_all_rows.csv` | 원본 1초 행 + A/B 각 센서별 점수·판정(`-1`은 계산 불가) |
| `07_if_overlap.csv` | A/B의 동일 시각 탐지 교집합·차집합 |

**해석 유의:** 후보는 고장 라벨이 아니며 `TOP5`는 점수 상위 비율입니다. A와 B의 유효 행 수가 다를 수 있습니다. 중복·강종변경·수리/재가동 영향은 결과 저장 후 외부 조업 정보와 대조하세요.
**저장 경로:** 위 7개 CSV 및 그래프 `if_*.png`는 모두 `pdm-project/data/sayletskin/isolation_forest_abc/`에 저장됩니다.

**참고:** 기존 저장 폴더에 이전 실행에서 만들어진 `if_C_*.png` 파일이 있다면 자동 삭제하지 않습니다. 필요 없으면 결과 확인 후 직접 삭제하세요.
